# Phase 6 - Geographic Aggregation by Borough

Goal: produce per-borough complaint volume + top categories + distinctive-term fingerprints (TF-IDF lift over corpus). These artifacts power the City Pulse and Cluster Atlas dashboard tabs.

**Pivot from the proposal:** the proposal targeted 71 community districts via spatial join, but the NYC Open Data community-districts dataset (`mzpm-a6vd`) is currently returning empty geometries. We use the 5-borough level instead. Borough is already a column in our 311 data so we skip the spatial join entirely. The 5-borough granularity also reads more clearly in the demo - Manhattan/Brooklyn/Queens/Bronx/Staten Island is universally recognizable.

Phase 2's `sample_2m_preprocessed.parquet` must be on Drive.

## Cell 1 - Bootstrap

In [ ]:
REPO_URL = 'https://github.com/george-gideon-S/cs-gy-6513-big-data-311-nlp.git'

from google.colab import drive
drive.mount('/content/drive')

import subprocess, os, sys
if not os.path.isdir('/content/project/.git'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/project'], check=True)
else:
    subprocess.run(['git', '-C', '/content/project', 'pull'], check=True)

if '/content/project' not in sys.path:
    sys.path.insert(0, '/content/project')

!pip install -r /content/project/requirements.txt -q

!apt-get install -y openjdk-11-jre-headless > /dev/null 2>&1
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

import nltk
for pkg in ['stopwords', 'wordnet', 'punkt', 'punkt_tab', 'omw-1.4']:
    nltk.download(pkg, download_dir='/root/nltk_data', quiet=True)

from src.spark_setup import get_spark
spark = get_spark(app_name='phase6-geo')
print('spark', spark.version, 'ready')

## Cell 2 - Load borough boundaries (already bundled in the repo)

In [ ]:
import json

geo_path = '/content/project/dashboard/assets/nyc_boroughs.geojson'
with open(geo_path) as f:
    geo = json.load(f)

print(f'features: {len(geo["features"])}')
for f in geo['features']:
    print(f'  {f["properties"]["name"]}: {f["geometry"]["type"]}')

## Cell 3 - Load preprocessed data + filter

We need rows with a non-null borough column and at least one token.

In [ ]:
from pyspark.sql import functions as F

in_path = '/content/drive/MyDrive/cs6513/sample_2m_preprocessed.parquet'
df = (
    spark.read.parquet(in_path)
    .filter(F.size('tokens') > 0)
    .filter(F.col('borough').isNotNull())
    # normalize borough strings - the dataset has mixed casing (BRONX vs Bronx)
    .withColumn('borough_norm', F.initcap(F.col('borough')))
    # keep only the 5 real boroughs (some rows have 'Unspecified' or empty)
    .filter(F.col('borough_norm').isin(['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']))
    .select('unique_key', 'label_canonical', 'borough_norm', 'tokens')
)
n = df.count()
print(f'rows after filter: {n:,}')

df.groupBy('borough_norm').count().orderBy(F.desc('count')).show()

## Cell 4 - Per-borough complaint volume + top categories

This drives the City Pulse choropleth tile.

In [ ]:
import pandas as pd

# volume per borough
volume = (
    df.groupBy('borough_norm').count()
    .withColumnRenamed('count', 'complaint_count')
    .toPandas()
    .sort_values('complaint_count', ascending=False)
)
print('borough complaint volumes:')
print(volume.to_string(index=False))

# top 5 categories per borough
cat_counts = (
    df.groupBy('borough_norm', 'label_canonical').count()
    .toPandas()
)
top_cats_per_borough = {}
for borough, sub in cat_counts.groupby('borough_norm'):
    sub = sub.sort_values('count', ascending=False).head(5)
    top_cats_per_borough[borough] = [
        (row['label_canonical'], int(row['count'])) for _, row in sub.iterrows()
    ]

print(f'\ntop 5 categories per borough:')
for b, cats in top_cats_per_borough.items():
    formatted = ', '.join(f'{n} ({c:,})' for n, c in cats)
    print(f'  {b}:')
    print(f'    {formatted}')

## Cell 5 - Per-borough TF-IDF lift fingerprints

Lift = (term frequency in borough) / (term frequency in whole corpus). High lift means a term is *distinctively used* in that borough vs the city average. Surfaces the language signature of each borough's complaint character.

In [ ]:
exploded = df.select('borough_norm', F.explode('tokens').alias('term'))

# tf per (borough, term)
by_borough = (
    exploded.groupBy('borough_norm', 'term').count()
    .withColumnRenamed('count', 'tf_borough')
)

# tf corpus-wide
by_corpus = (
    exploded.groupBy('term').count()
    .withColumnRenamed('count', 'tf_corpus')
)
corpus_total = by_corpus.agg(F.sum('tf_corpus')).collect()[0][0]
by_corpus = by_corpus.withColumn('tf_corpus_norm', F.col('tf_corpus') / F.lit(corpus_total))

# borough totals for normalization
borough_totals = (
    by_borough.groupBy('borough_norm').agg(F.sum('tf_borough').alias('b_total'))
)

# join + compute lift
lift_df = (
    by_borough.join(by_corpus, on='term', how='left')
    .join(borough_totals, on='borough_norm', how='left')
    .withColumn('tf_borough_norm', F.col('tf_borough') / F.col('b_total'))
    .withColumn('lift', F.col('tf_borough_norm') / F.col('tf_corpus_norm'))
    # require >= 100 occurrences in borough so we dont surface typos
    .filter(F.col('tf_borough') >= 100)
)

lift_pdf = lift_df.select('borough_norm', 'term', 'tf_borough', 'lift').toPandas()
print(f'computed lift for {len(lift_pdf):,} (borough, term) pairs')

## Cell 6 - Top 15 distinctive terms per borough

These become the per-borough word clouds in the Cluster Atlas tab.

In [ ]:
fingerprints = {}
for borough, sub in lift_pdf.groupby('borough_norm'):
    top = sub.nlargest(15, 'lift')
    fingerprints[borough] = [
        (row['term'], round(float(row['lift']), 2), int(row['tf_borough']))
        for _, row in top.iterrows()
    ]

print('per-borough distinctive-term fingerprints:')
for b, terms in fingerprints.items():
    print(f'\n  {b}:')
    for term, lift, count in terms[:8]:
        print(f'    {term:25s}  lift={lift:5.2f}  (n={count:,})')

## Cell 7 - Save artifacts for the dashboard

In [ ]:
import datetime

# 1. borough volume + top categories
volume_dict = {
    row['borough_norm']: {
        'count': int(row['complaint_count']),
        'top_categories': top_cats_per_borough[row['borough_norm']],
    }
    for _, row in volume.iterrows()
}
with open('/content/project/dashboard/assets/borough_volume.json', 'w') as f:
    json.dump(volume_dict, f, indent=2, default=str)
print('saved borough_volume.json')

# 2. per-borough fingerprints
with open('/content/project/dashboard/assets/borough_fingerprints.json', 'w') as f:
    json.dump(fingerprints, f, indent=2, default=str)
print('saved borough_fingerprints.json')

# 3. summary metadata
summary = {
    'phase': 6,
    'computed_at': datetime.datetime.utcnow().isoformat() + 'Z',
    'rows_aggregated': int(n),
    'aggregation_unit': 'borough',
    'n_boroughs': len(fingerprints),
    'volumes': [
        {'borough': row['borough_norm'], 'count': int(row['complaint_count'])}
        for _, row in volume.iterrows()
    ],
    'note': 'pivoted from community-districts to boroughs because mzpm-a6vd dataset returns empty geometries currently',
}
with open('/content/project/dashboard/assets/geo_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print('saved geo_summary.json')

## Cell 8 - Push artifacts to GitHub

Commits the borough JSONs to the repo. The geojson is already in the repo (committed locally and pushed).

In [ ]:
from src.colab_git import commit_artifacts
commit_artifacts(message='phase 6: per-borough volume + fingerprints')

## Phase 6 - Done when

- Cell 4 prints all 5 boroughs with reasonable volumes (Brooklyn / Manhattan / Queens dominate, Staten Island lowest).
- Cell 6 prints distinctive terms that pass an eyeball test (e.g., Manhattan leads with terms about commercial / construction; Bronx with rodent / heating; Staten Island with parking).
- `dashboard/assets/{nyc_boroughs.geojson, borough_volume.json, borough_fingerprints.json, geo_summary.json}` all in the repo.
- Cell 8 successfully pushes to GitHub.

Save the print as `PRINT 8.pdf` and drop in the project directory. Phase 6 unlocks the City Pulse dashboard tab.